# Tutorial 7: The Training Toolkit — Extractor, Diagnostics & Entropy Decay

Spectral-Env-Core Pro ships three importable training components that solve the hardest problems in financial RL:

1. **SpectralExtractor** — shared-encoder architecture for multi-asset observations
2. **DiagnosticsCallback** — answers "why is my agent failing" not just "is it failing"
3. **EntropyCoefficientCallback** — controlled exploration decay

This tutorial shows how they work together in a production training pipeline.

---

In [ ]:
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback

from spectral_env_core import (
    SpectralTradingEnv,
    SpectralExtractor,
    DiagnosticsCallback,
    EntropyCoefficientCallback,
    estimate_params_multi,
)

## 1. SpectralExtractor — Why a Custom Extractor Matters

### The problem with default MLP policies

SB3's `MlpPolicy` treats the entire observation as a flat vector. For a 5-asset environment with `lookback_window=30`, that's `5×30 + 9 = 159` inputs. The MLP has no structural understanding that:

- Inputs 0-29 are Asset 0's price history
- Inputs 30-59 are Asset 1's price history
- ...
- The last 9 values are portfolio metadata

It treats them all identically. Learning to recognise "momentum" in one asset doesn't transfer to another — the network must learn the same pattern 5 times independently.

### SpectralExtractor's architecture

```
Observation (159 dims)
    │
    ├── price_windows (5 × 30 = 150 dims)
    │       │
    │       ▼
    │   [Shared price_net]  ← same weights for all 5 assets
    │   Linear(30→64) → LayerNorm → ReLU → Linear(64→32) → LayerNorm → ReLU
    │       │
    │       ▼
    │   5 × 32 = 160 dim price embeddings
    │
    └── meta_features (9 dims: cash, shares×5, portfolio, exit_cost, time_remaining)
            │
            ▼
        [meta_net]
        Linear(9→16) → ReLU
            │
            ▼
        16 dim meta embedding

Concatenate → 176 dims → Actor [64, 64] / Critic [128, 128, 64]
```

**Key insight:** the `price_net` uses shared weights. A momentum pattern learned from AAPL's window automatically applies to MSFT's window too. The network learns *what temporal patterns look like*, not *where in the input vector to find them*.

In [ ]:
# Configure
env_kwargs = {
    "num_assets": 3,
    "num_steps": 252,
    "initial_price": [180.0, 300.0, 100.0],
    "drift": [0.15, 0.22, 0.08],
    "volatility": [0.28, 0.32, 0.15],
    "garch_alpha": 0.09,
    "garch_beta": 0.84,
    "jump_intensity": 2.0,
    "jump_mean": -0.03,
    "jump_std": 0.06,
    "starting_cash": 100_000,
    "max_shares": 500,
    "max_trade_size": 50,
    "correlation": np.array([
        [1.0, 0.6, 0.2],
        [0.6, 1.0, 0.3],
        [0.2, 0.3, 1.0],
    ]),
}

# SpectralExtractor plugs directly into policy_kwargs
policy_kwargs = dict(
    log_std_init=-1.0,
    features_extractor_class=SpectralExtractor,
    features_extractor_kwargs=dict(
        num_assets=3,
        lookback_window=30,
        asset_embed_dim=32,
        meta_embed_dim=16,
    ),
    net_arch=dict(
        pi=[64, 64],        # actor
        vf=[128, 128, 64],  # deeper critic for noisy value landscapes
    ),
)

print(f"Extractor output: {3*32 + 16} = 112 dims")
print(f"Actor:  112 → 64 → 64 → 3 actions")
print(f"Critic: 112 → 128 → 128 → 64 → 1 value")

## 2. DiagnosticsCallback — Know *Why* Training Fails

SB3's built-in logging tells you mean reward and episode length. That's not enough. When reward stalls you need to know:

| Question | Metric |
|---|---|
| Is the agent trading or holding cash? | `mean_abs_action`, `dead_zone_frac` |
| Is it paying too much in fees? | `mean_friction` |
| Is it going bankrupt? | `truncation_rate` |
| Can the critic predict returns? | `explained_variance` |
| Which regime is harder? | `regime_fracs` |
| Is entropy decaying correctly? | `ent_coef` |

`DiagnosticsCallback` captures all of this at regular intervals and writes to `diagnostics.npz`.

It also **automatically syncs VecNormalize statistics** between training and eval environments — solving a common SB3 pitfall where eval rewards are wildly inaccurate during early training.

In [ ]:
# Set up parallel training with full diagnostics
train_vec = make_vec_env(SpectralTradingEnv, n_envs=4, env_kwargs=env_kwargs, vec_env_cls=SubprocVecEnv)
train_vec = VecNormalize(train_vec, norm_obs=True, norm_reward=False, clip_obs=10.0)

eval_env = make_vec_env(SpectralTradingEnv, n_envs=1, env_kwargs=env_kwargs)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.0, training=False)

# DiagnosticsCallback handles VecNormalize sync + all metrics
diag = DiagnosticsCallback(
    eval_env=eval_env,
    train_env=train_vec,
    n_eval_episodes=10,
    eval_freq=4096,        # every 4096 total timesteps
    log_path="./demo_logs",
)

print("DiagnosticsCallback configured:")
print(f"  Eval every {4096:,} steps")
print(f"  10 episodes per checkpoint")
print(f"  Auto-syncs VecNormalize stats")
print(f"  Writes: ./demo_logs/diagnostics.npz")

## 3. EntropyCoefficientCallback — Controlled Exploration

Financial RL has a unique exploration challenge: the action dead zone.

Actions with `|a| < 0.05` are treated as HOLD (no trade). If entropy is too low at the start, the policy stays inside this dead zone indefinitely — the agent never learns to trade. If entropy is too high, the agent trades maximally and churns through transaction costs.

The solution: start with moderate entropy to push past the dead zone, then decay linearly so the policy sharpens as it learns.

```
ent_coef: 0.005 ──────────────────── 0.001
                start                 end
           (explore)              (exploit)
```

In [ ]:
# Entropy decay: explore early, sharpen late
ent_cb = EntropyCoefficientCallback(start=0.005, end=0.001)

# Build model with all components
model = PPO(
    "MlpPolicy",
    train_vec,
    verbose=1,
    device="cpu",
    n_steps=2048,
    batch_size=128,
    n_epochs=10,
    learning_rate=3e-5,
    gamma=0.995,
    gae_lambda=0.90,
    clip_range=0.2,
    target_kl=0.05,
    ent_coef=0.005,   # initial value — callback handles decay
    vf_coef=0.75,
    policy_kwargs=policy_kwargs,
)

print(f"\nTotal parameters: {sum(p.numel() for p in model.policy.parameters()):,}")
print(f"Training with: SpectralExtractor + DiagnosticsCallback + EntropyCoefficientCallback")

In [ ]:
# Short training run to demonstrate the pipeline
print("\nTraining for 100K steps (demo)...\n")
model.learn(
    total_timesteps=100_000,
    callback=[ent_cb, diag],
)

train_vec.close()
eval_env.close()

In [ ]:
# Load and inspect diagnostics
import os
data = np.load(os.path.join("./demo_logs", "diagnostics.npz"))

print(f"Checkpoints logged: {len(data['timesteps'])}")
print(f"\nMetrics available:")
for key in sorted(data.files):
    print(f"  {key}: shape {data[key].shape}")

print(f"\nLatest checkpoint:")
print(f"  Reward:      {data['mean_reward'][-1]:+.2f} ± {data['std_reward'][-1]:.2f}")
print(f"  Dead zone:   {data['dead_zone_frac'][-1]*100:.1f}% of actions zeroed")
print(f"  Friction:    ${data['mean_friction'][-1]:.2f} avg per episode")
print(f"  Trunc rate:  {data['truncation_rate'][-1]*100:.1f}%")
print(f"  Exp. var:    {data['explained_var'][-1]:.4f}")
print(f"  Ent coef:    {data['ent_coef'][-1]:.5f}")

## Reading the Diagnostics

Here's how to interpret each metric:

| Metric | Healthy Range | Problem Signal |
|---|---|---|
| `dead_zone_frac` | 10-25% (agent trades selectively) | >80% = not exploring; <5% = churning |
| `mean_friction` | Growing slowly, proportional to trades | Exploding upward = overtrading |
| `truncation_rate` | 0-5% | >20% = agent is going bankrupt |
| `explained_variance` | >0.3 and rising | Near 0 or negative = critic broken |
| `ent_coef` | Smoothly decaying | Flat = callback not firing |
| `mean_abs_action` | 0.15-0.40 | <0.05 = stuck in dead zone; >0.8 = maxing out |

Use the included `logs/examine.ipynb` dashboard to visualise all metrics across training runs.

---

## The Full Import Surface

```python
from spectral_env_core import (
    # Environment
    SpectralTradingEnv,
    SpectralRenderWrapper,

    # Feature extraction
    SpectralExtractor,

    # Training callbacks
    DiagnosticsCallback,
    EntropyCoefficientCallback,

    # Parameter calibration
    estimate_params,
    estimate_params_multi,
)
```

Everything you need to go from ticker to trained agent in a single package.